In [1]:
# function to calculate the autopet3_py metrics 

# Method to get the lesion metrics and save as a csv

In [2]:

import os
import glob
import json
import pickle

import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk

import numpy as np
import nibabel as nib
import pathlib as plb
import cc3d
import csv
import sys

import argparse

def handle_arguments():
    """
    Handle command line arguments for processing images.
    """
    parser = argparse.ArgumentParser(description='Process probability analysis with various options')
    parser.add_argument('-k', '--key', type=str, default='*', 
                       help='Key pattern for file matching (default: *)')
    parser.add_argument('-l', '--lesion-only', action='store_true', 
                       help='Process only lesion data, skip CTImage object creation')
    parser.add_argument('-f', '--fold', type=str, default='comb', 
                       help='Fold name/number (default: 1)')
    parser.add_argument('-s', '--skip-existing', action='store_true',
                       help='Skip processing files that have already been processed', default=False)
    parser.add_argument('-i', '--image-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Public_Datasets/Autopet_III_nnunet_raw/Dataset888_AutoPet/imagesTr/", help='Directory containing images (default: ../imagesTr/)')
    parser.add_argument('-m', '--mets-only', action='store_true', 
                       help='Process only mets data, skip lesion-only data')
    parser.add_argument('-ld', '--label-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Results/", help='Directory containing labels (default: ../labelsTr/)') 
    parser.add_argument('-b', '--bone-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Results/Biratal/bone-metastasis/totalsegmentator/bone_segmentations_fold1/", help='Directory containing bone segmentations (default: ../totalsegmentator/bone_segmentations_fold1/)')

    parser.add_argument('-p', '--pred-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset888_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1_ens_comb/test_predictions/", 
                        help='Directory containing prediction files (default: //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset888_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1_ens_comb/test_predictions/)')

    parser.add_argument('-u', '--uncertainty-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset888_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1_ens_comb/uncertainty_maps/", 
                        help='Directory containing uncertainty maps (default: //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset888_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1_ens_comb/uncertainty_maps/)')
    parser.add_argument('-o', '--save-dir', type=str, default=args.pred_dir, 
                        help='Directory to save lesion analysis results (default: ./lesion_analysis_fold)')
    args = parser.parse_args()
    return args


def nii2numpy(nii_path):
    # input: path of NIfTI segmentation file, output: corresponding numpy array and voxel_vol in ml
    mask_nii = nib.load(str(nii_path))
    mask = mask_nii.get_fdata()
    pixdim = mask_nii.header['pixdim']   
    voxel_vol = pixdim[1]*pixdim[2]*pixdim[3]/1000
    return mask, voxel_vol


def con_comp(seg_array):
    # input: a binary segmentation array output: an array with seperated (indexed) connected components of the segmentation array
    connectivity = 18
    conn_comp = cc3d.connected_components(seg_array, connectivity=connectivity)
    return conn_comp



def false_pos_pix(gt_array,pred_array):
    # compute number of voxels of false positive connected components in prediction mask
    pred_conn_comp = con_comp(pred_array)
    
    false_pos = 0
    false_detected_lesion_count = 0
    
    for idx in range(1,pred_conn_comp.max()+1):
        comp_mask = np.isin(pred_conn_comp, idx)
        if (comp_mask*gt_array).sum() == 0:
            false_pos = false_pos+comp_mask.sum()
            false_detected_lesion_count += 1
    return false_pos, false_detected_lesion_count, pred_conn_comp.max()

def false_neg_pix(gt_array,pred_array):
    # compute number of voxels of false negative connected components (of the ground truth mask) in the prediction mask
    gt_conn_comp = con_comp(gt_array)
    
    false_neg = 0
    false_missed_lesion_count = 0
    
    for idx in range(1,gt_conn_comp.max()+1):
        comp_mask = np.isin(gt_conn_comp, idx)
        if (comp_mask*pred_array).sum() == 0:
            false_neg = false_neg+comp_mask.sum()
            false_missed_lesion_count += 1
            
    return false_neg, false_missed_lesion_count, gt_conn_comp.max()

def dice_score(mask1,mask2):
    # compute foreground Dice coefficient
    overlap = (mask1*mask2).sum()
    sum = mask1.sum()+mask2.sum()
    dice_score = 2*overlap/sum
    return dice_score



def compute_metrics(nii_gt_path, nii_pred_path):
    # main function
    gt_array, voxel_vol = nii2numpy(nii_gt_path)
    pred_array, voxel_vol = nii2numpy(nii_pred_path)
    false_neg_vol, false_missed_lesion_count, gt_lesion_count= false_neg_pix(gt_array, pred_array)
    false_neg_vol = false_neg_vol*voxel_vol

    false_pos_vol, false_detected_lesion_count, pred_lesion_count = false_pos_pix(gt_array, pred_array)
    false_pos_vol = false_pos_vol*voxel_vol
    
    dice_sc = dice_score(gt_array,pred_array)

    return dice_sc, false_pos_vol, false_neg_vol, false_missed_lesion_count, false_detected_lesion_count



In [3]:
import tqdm
nnUNet_results = os.environ['nnUNet_results']
dataset_name = "Dataset999_AutoPet"
fold = "fold_1"
testing_split = "test_predictions"

pred_dir="{nnUNet_results}/{dataset_name}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{testing_split}"
gt_dir = "{nnUNet_results}/test/gt/"

out_dir="{nnUNet_results}/{dataset_name}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{testing_split}/lesion_metrics.csv"


In [4]:
prediction_file_names = [x for x in os.listdir(pred_dir.format(nnUNet_results=nnUNet_results,
 dataset_name=dataset_name, 
 fold=fold, 
 testing_split=testing_split)) if x.endswith('.nii.gz')]

gt_files = [os.path.join(gt_dir.format(nnUNet_results=nnUNet_results), x) for x in prediction_file_names]  # Match GT files to prediction files based on filename
prediction_files = [os.path.join(pred_dir.format(nnUNet_results=nnUNet_results,
 dataset_name=dataset_name,
 fold=fold,
 testing_split=testing_split), x) for x in prediction_file_names]
 

In [ ]:
def read_summary_file(path, fold, validation): 
    with open(path.format(fold=fold, validation=validation), "r") as f:
        return json.load(f)
    
def load_dataset_folds(dataset_num, folds=[f'fold_{i}' for i in range(0, 11)], validation="validation_279"):
    path = DEFAULT_SUMMARY_PATH.format(
        input_dataset=f"Dataset{dataset_num}_AutoPet",
        fold="{fold}",
        validation="{validation}"
    )
    

    summary_dfs = []
    fdg_dfs = []
    psma_dfs = []
    
    for i in folds:
        filepath = path.format(fold=i, validation=validation)
        try:
            fold_data = read_summary_file(path, fold=i, validation=validation)
            print(f"Successfully read fold {i} for dataset {dataset_num}")
            # --- Per-case metrics, split by tracer ---
            fdg_rows = []
            psma_rows = []
            lesion_metrics_path = filepath.replace("summary.json", "lesion_metrics.csv")
            lesion_metrics_df = pd.read_csv(lesion_metrics_path)

            for case in fold_data['metric_per_case']:
                metrics = case['metrics']['1']
                pred_file = case['prediction_file']
                case_name = os.path.basename(pred_file).replace('.nii.gz', '')
                
                # now for this case, find the actual FP Vol from the lesion_metrics file
                # get for that case
                lesion_subdf = lesion_metrics_df[lesion_metrics_df['filename'] == case_name]

                # only add to dice if 

                row = {
                    "Dice": metrics['Dice'],
                    "FP":   lesion_subdf['false_pos_vol'].values[0] if not lesion_subdf.empty else np.nan,
                    "FN":   lesion_subdf['false_neg_vol'].values[0] if not lesion_subdf.empty else np.nan,
                }
                
                if case_name.lower().startswith('fdg'):
                    fdg_rows.append(row)
                elif case_name.lower().startswith('psma'):
                    psma_rows.append(row)
            
            def make_summary_row(rows, fold_label):
                if not rows:
                    return None
                df = pd.DataFrame(rows)
                return pd.DataFrame({
                    "fold":          [fold_label],
                    "Dice":          [round(df['Dice'].mean(), 4)],
                    "FP (voxels)":   [int(df['FP'].mean())],
                    "FN (voxels)":   [int(df['FN'].mean())],
                    "n_cases":       [len(rows)],
                })
            
            # Overall summary from foreground_mean
            df_summary = pd.DataFrame({
                "fold":        [str(i)],
                "Dice":        [round(fold_data['foreground_mean']['Dice'], 4)],
                "FP (voxels)": [int(fold_data['foreground_mean']['FP'])],
                "FN (voxels)": [int(fold_data['foreground_mean']['FN'])],
                "n_cases":     [len(fold_data['metric_per_case'])],
            })
            summary_dfs.append(df_summary)
            
            row_fdg  = make_summary_row(fdg_rows,  str(i))
            row_psma = make_summary_row(psma_rows, str(i))
            if row_fdg  is not None: fdg_dfs.append(row_fdg)
            if row_psma is not None: psma_dfs.append(row_psma)
            
        except Exception as e:
            print(f"Could not read fold {i} for dataset {dataset_num}")
            print(f"Expected path: {filepath}")
            print(f"Error: {e}")
    
    summary_df = pd.concat(summary_dfs, ignore_index=True).sort_values('fold')
    fdg_df     = pd.concat(fdg_dfs,     ignore_index=True).sort_values('fold') if fdg_dfs  else None
    psma_df    = pd.concat(psma_dfs,    ignore_index=True).sort_values('fold') if psma_dfs else None
    
    return summary_df, fdg_df, psma_df


In [11]:
import os

DEFAULT_SUMMARY_PATH = "{nnUNet_results}/{input_dataset}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{validation}/summary.json"\
    .format(nnUNet_results=os.environ['nnUNet_results'], input_dataset="{input_dataset}", fold="{fold}", validation="{validation}") 
    
print(DEFAULT_SUMMARY_PATH)

/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/{input_dataset}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{validation}/summary.json


In [12]:
load_dataset_folds(dataset_num=999, folds=[f'fold_{i}' for i in range(0, 11)], validation="validation_279")

Successfully read fold fold_0 for dataset 999
Could not read fold fold_0 for dataset 999
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/validation_279/summary.json
Error: cannot convert float NaN to integer
Successfully read fold fold_1 for dataset 999
Could not read fold fold_1 for dataset 999
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1/validation_279/summary.json
Error: cannot convert float NaN to integer
Successfully read fold fold_2 for dataset 999
Could not read fold fold_2 for dataset 999
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_2/validation_279/summary.json
Error: cannot conver

(     fold    Dice  FP (voxels)  FN (voxels)  n_cases
 0  fold_0  0.3215          430         7807      247
 1  fold_1  0.5730         2049         3480      247
 2  fold_2  0.5937         2501         3265      247
 3  fold_3  0.6167         2495         2767      247
 4  fold_4  0.6090         1726         3087      247
 5  fold_5  0.6348         1853         3256      247
 6  fold_6  0.6295         1758         2977      247
 7  fold_7  0.6347         2084         2876      247
 8  fold_8  0.6523         2213         2464      247
 9  fold_9  0.6587         2444         1970      247,
 None,
 None)

## Once you have the lesion metrics, processing to generate the tables 
This is to generate the figures for the validation table figures. So the Dice, FP Vol, FN Vol across disease 

In [8]:
df = pd.read_csv('../fdg_metadata.csv')
def convert_location_to_case_name(file_locations):
    case_names = []
    for path in file_locations:
        parts = path.split('/')
        # Extract patient ID from PETCT_xxxx
        patient_folder = parts[2]  # e.g., 'PETCT_0011f3deaf'
        patient_id = patient_folder.replace('PETCT_', '')
        
        # Extract scan folder (date-NA-scan_type-number)
        scan_folder = parts[3]  # e.g., '03-23-2003-NA-PET-CT Ganzkoerper...-10445'
        
        # Create CSV filename
        case_name = f"fdg_{patient_id}_{scan_folder}"
        case_names.append(case_name)
    
    return np.array(case_names)

def add_diagnosis(df): 
    fdg_metadata = pd.read_csv('fdg_metadata.csv')
    fdg_metadata['file'] = convert_location_to_case_name(fdg_metadata['File Location'].values)
    df = df.merge(
        fdg_metadata[['file', 'diagnosis']].drop_duplicates(subset=['file', 'diagnosis']),
        on='file',
        how='left'
        )
    
    # all PSMA cases are prostate cancer
    df.loc[df['file'].str.contains('psma', case=False, na=False), 'diagnosis'] = 'PROSTATE_CANCER'
    return df


nnUNet_results = os.environ['nnUNet_results']
testing_splits = "validation_279"
BASE_PATH = "{nnUNet_results}/{dataset_name}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{testing_split}/lesion_metrics.csv"

def get_lesion_metrics(dataset, fold):
    fpath = BASE_PATH.format(nnUNet_results=nnUNet_results, dataset_name=dataset, fold=fold, testing_split=testing_splits)
    tempdf = pd.read_csv(fpath)
    tempdf['filename'] = tempdf['filename'].str.replace('.nii.gz', '')
    return tempdf

temp = get_lesion_metrics("Dataset999_AutoPet", "fold_0")  

temp

,filename,dice_score,false_pos_vol,false_neg_vol,false_missed_lesions,false_detected_lesions
0,psma_a672c1e6a35c21b8_2019-07-20,0.906067,0.000000,1.658795,1,0
1,psma_527f1b3f98ecf49c_2015-11-23,0.212987,0.024449,4.914281,4,1
2,psma_ad7bc1aa5b586751_2018-09-14,0.712034,0.000000,3.085358,3,0
3,fdg_a37c4e231f_12-13-2002-NA-PET-CT Ganzkoerpe...,NaN,0.000000,0.000000,0,0
4,psma_8b3f81670b02a757_2020-11-02,0.386485,3.566408,38.235215,42,3
...,...,...,...,...,...,...
242,fdg_a41d59682f_09-29-2006-NA-PET-CT Ganzkoerpe...,NaN,75.479677,0.000000,0,7
243,psma_fe15e4b0571c9233_2018-02-18,0.156373,5.872133,206.055472,37,3
244,fdg_7b477e7e0d_07-26-2002-NA-PET-CT Ganzkoerpe...,0.406505,0.000000,4.093096,11,0
245,psma_2d635a895be772d5_2019-08-10,0.611410,4.669789,36.184755,63,19


In [9]:
lesion_metrics_df = pd.read_csv('../lesion_metrics.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../lesion_metrics.csv'